In [1]:
# ==================== CELL 1: 修复路径匹配并补全 manifest.csv ====================
import os, glob, yaml
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
import imagehash
from IPython.display import Image as IPyImage, display

DATASET_DIR = "11106655_pallet_part00_phase20"
RUNS_CSV_PATH = os.path.join(DATASET_DIR, "pilot", "runs_pilot.csv")
FIG_DIR = "report_figures"
os.makedirs(FIG_DIR, exist_ok=True)

manifest = pd.read_csv(os.path.join(DATASET_DIR, "manifest.csv"))
runs = pd.read_csv(RUNS_CSV_PATH)

# 兼容 Windows 反斜杠和正斜杠，统一提取纯文件名（小写去除空格）
manifest["_key"] = manifest["source_json"].astype(str).str.replace("\\", "/").apply(os.path.basename).str.strip().str.lower()
runs["_key"] = runs["export_file"].astype(str).str.replace("\\", "/").apply(os.path.basename).str.strip().str.lower()

print("manifest 里的样本文件名:", manifest["_key"].unique()[:5])
print("runs_pilot 里的样本文件名:", runs["_key"].unique()[:5])

merge_cols = [
    "_key", "run_id", "mode", "surface_mode", "seed",
    "jitterRatio", "materialVariation",
    "alphaMin", "alphaMax", "alphaStep",
    "betaMin", "betaMax", "betaStep",
    "radiusFactorMin", "radiusFactorMax", "radiusFactorStep"
]

# 先清除上一次运行残留的空列，防止 merge 冲突
manifest = manifest.drop(columns=[c for c in merge_cols if c != "_key" and c in manifest.columns])
manifest = manifest.merge(runs[merge_cols], on="_key", how="left")
manifest.drop(columns=["_key"], inplace=True)

manifest.to_csv(os.path.join(DATASET_DIR, "manifest.csv"), index=False)
print(f"manifest.csv 已更新，共 {len(manifest)} 行。缺失 mode 的行数: {manifest['mode'].isna().sum()}")
# 找出 manifest 里有，但 runs_pilot.csv 里漏登记的文件名及各自的图片数量
missing_summary = manifest[manifest['mode'].isna()]['source_json'].value_counts()
print("以下文件在 runs_pilot.csv 中缺失记录（文件名: 图片数）：")
print(missing_summary)

manifest 里的样本文件名: ['m1_r01_pilot.json' 'm1_r02_pilot.json' 'm2_r01_k2_pilot.json'
 'm2_r02_k3_pilot.json' 'm2_r03_k4_pilot.json']
runs_pilot 里的样本文件名: ['m1_r01_pilot.json' 'm1_r02_pilot.json' 'm2_r01_k2_pilot.json'
 'm2_r02_k3_pilot.json' 'm2_r03_k4_pilot.json']
manifest.csv 已更新，共 258 行。缺失 mode 的行数: 0
以下文件在 runs_pilot.csv 中缺失记录（文件名: 图片数）：
Series([], Name: count, dtype: int64)


In [2]:
# ==================== CELL 2: 遍历解析所有图像与双标签 ====================
with open(os.path.join(DATASET_DIR, "data.yaml"), "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

resolutions = set()
instances_records = []
r10_errors = []
poly_box_diffs_px = []

for _, row in manifest.iterrows():
    img_name = row["image"]
    stem = os.path.splitext(img_name)[0]
    img_path = os.path.join(DATASET_DIR, "images", img_name)
    box_path = os.path.join(DATASET_DIR, "labels", f"{stem}.txt")
    seg_path = os.path.join(DATASET_DIR, "labels_seg", f"{stem}.txt")

    if not os.path.exists(img_path):
        r10_errors.append(f"Missing image: {img_name}")
        continue
    with Image.open(img_path) as im:
        w_px, h_px = im.size
        resolutions.add((w_px, h_px))

    if not os.path.exists(box_path) or not os.path.exists(seg_path):
        r10_errors.append(f"Missing label files for: {stem}")
        continue

    with open(box_path) as fb, open(seg_path) as fs:
        box_lines = [l.strip() for l in fb if l.strip()]
        seg_lines = [l.strip() for l in fs if l.strip()]

    if len(box_lines) != len(seg_lines) or not (1 <= len(box_lines) <= 5):
        r10_errors.append(f"Line count mismatch or invalid count in {stem}: box={len(box_lines)}, seg={len(seg_lines)}")

    for b_line, s_line in zip(box_lines, seg_lines):
        b_parts = list(map(float, b_line.split()))
        s_parts = list(map(float, s_line.split()))
        cls_b, cx, cy, bw, bh = int(b_parts[0]), b_parts[1], b_parts[2], b_parts[3], b_parts[4]
        cls_s = int(s_parts[0])
        pts = np.array(s_parts[1:], dtype=float).reshape(-1, 2)

        # R10 格式与坐标范围检查
        if cls_b != 0 or cls_s != 0:
            r10_errors.append(f"Non-zero class ID in {stem}")
        if not (0 <= cx <= 1 and 0 <= cy <= 1 and 0 < bw <= 1 and 0 < bh <= 1):
            r10_errors.append(f"Box out of [0,1] in {stem}")
        if len(pts) < 3 or np.any(pts < 0) or np.any(pts > 1):
            r10_errors.append(f"Polygon invalid in {stem}")

        # Box 与 Polygon 外接矩形像素误差 (R10: <= 2 px)
        bx1, by1 = (cx - bw / 2) * w_px, (cy - bh / 2) * h_px
        bx2, by2 = (cx + bw / 2) * w_px, (cy + bh / 2) * h_px
        px1, py1 = pts[:, 0].min() * w_px, pts[:, 1].min() * h_px
        px2, py2 = pts[:, 0].max() * w_px, pts[:, 1].max() * h_px
        max_diff = max(abs(bx1 - px1), abs(by1 - py1), abs(bx2 - px2), abs(by2 - py2))
        poly_box_diffs_px.append(max_diff)

        # R8 边缘截断判定（边界距离图像边缘 <= 2 像素）
        is_truncated = (bx1 <= 2.0) or (by1 <= 2.0) or (bx2 >= w_px - 2.0) or (by2 >= h_px - 2.0)

        instances_records.append({
            "image": img_name,
            "mode": row["mode"],
            "cx": cx,
            "cy": cy,
            "area_pct": bw * bh * 100.0,
            "short_side_px": min(bw * w_px, bh * h_px),
            "truncated": is_truncated
        })

df_inst = pd.DataFrame(instances_records)
print(f"解析完成: {len(manifest)} 张图像, {len(df_inst)} 个实例")

解析完成: 258 张图像, 376 个实例


In [ ]:
# ==================== CELL 3: R2 (每图实例数分布) & R4 (生成模式配比) ====================
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image as IPyImage, display

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 1. R2: 每图实例数分布 (1~5 个实例的图像占比)
inst_counts = manifest["n_instances"].value_counts().reindex([1, 2, 3, 4, 5], fill_value=0)
inst_shares = inst_counts / len(manifest) * 100
targets_r2 = [70, 10, 8, 6, 6]  # 指导书 R2 目标配比 (%)

x = np.arange(5)
bars_meas = axes[0].bar(x - 0.18, inst_shares, width=0.35, label="Measured (%)", color="#3B82F6")
bars_targ = axes[0].bar(x + 0.18, targets_r2, width=0.35, label="Target (%)", color="#94A3B8")

axes[0].set_xticks(x)
axes[0].set_xticklabels(["1", "2", "3", "4", "5"])
axes[0].set_title("R2: Instances per Image Distribution")
axes[0].set_xlabel("Instances per Image")
axes[0].set_ylabel("Share of Images (%)")
axes[0].set_ylim(0, max(max(inst_shares), 75) + 10)
axes[0].legend()

# 在柱状图上方标注具体百分比数值
for bar in bars_meas:
    h = bar.get_height()
    axes[0].annotate(f"{h:.1f}%", xy=(bar.get_x() + bar.get_width() / 2, h),
                     xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=8)

# 2. R4: 各生成模式图像数量 (M1–M5)
mode_counts = manifest["mode"].value_counts().reindex(["M1", "M2", "M3", "M4", "M5"], fill_value=0)
bars_mode = axes[1].bar(mode_counts.index, mode_counts.values, color="#10B981", width=0.5)

axes[1].axhline(100, color="red", linestyle="--", label="Auto Min (100, 10%)")
axes[1].axhline(300, color="orange", linestyle="--", label="M2 Target (300, 30%)")
axes[1].set_title("R4: Images per Generation Mode")
axes[1].set_xlabel("Generation Mode")
axes[1].set_ylabel("Image Count")
axes[1].set_ylim(0, max(max(mode_counts.values), 320) + 40)
axes[1].legend()

# 在柱状图上方标注具体张数和占比
for bar in bars_mode:
    h = int(bar.get_height())
    pct = h / len(manifest) * 100
    axes[1].annotate(f"{h}\n({pct:.1f}%)", xy=(bar.get_x() + bar.get_width() / 2, h),
                     xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
out_path = f"{FIG_DIR}/R2_R4_distributions.png"
plt.savefig(out_path, dpi=150)
plt.close()

# 打印详细数值统计表
print("=== R2 每图实例数统计 ===")
for k in [1, 2, 3, 4, 5]:
    print(f"  {k} 个实例: {inst_counts[k]} 张 ({inst_shares[k]:.2f}%, 目标 {targets_r2[k-1]}% ±3%)")

print("\n=== R4 生成模式分布统计 ===")
for m in ["M1", "M2", "M3", "M4", "M5"]:
    cnt = mode_counts[m]
    print(f"  {m}: {cnt} 张 ({cnt / len(manifest) * 100:.2f}%)")

# 安全内嵌显示图表（不会卡死）
display(IPyImage(filename=out_path))

In [ ]:
# ==================== CELL 4: R5 尺度分箱 & R6 24视角网格覆盖 ====================
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# R5: 面积三档柱状图
small_pct = (df_inst["area_pct"] < 1.0).mean() * 100
med_pct = ((df_inst["area_pct"] >= 1.0) & (df_inst["area_pct"] <= 10.0)).mean() * 100
large_pct = (df_inst["area_pct"] > 10.0).mean() * 100
r5_meas = [small_pct, med_pct, large_pct]
r5_targ = [30, 40, 30]
xb = np.arange(3)
axes[0].bar(xb - 0.18, r5_meas, width=0.35, label="Measured (%)", color="#6366F1")
axes[0].bar(xb + 0.18, r5_targ, width=0.35, label="Target (%)", color="#94A3B8")
axes[0].set_xticks(xb)
axes[0].set_xticklabels(["Small (<1%)", "Medium (1-10%)", "Large (>10%)"])
axes[0].set_title("R5: Scale Ladder Distribution")
axes[0].set_ylabel("Share of Instances (%)")
axes[0].legend()

# R6: 8x3 Azimuth x Elevation 视角覆盖网格 (自动模式 M1, M3-M5)
auto_runs = runs[runs["mode"].isin(["M1", "M3", "M4", "M5"])]
grid_r6 = np.zeros((3, 8), dtype=int)
for _, r in auto_runs.iterrows():
    alphas = np.arange(r["alphaMin"], r["alphaMax"] + 1e-5, r["alphaStep"]) % 360
    betas = np.arange(r["betaMin"], r["betaMax"] + 1e-5, r["betaStep"])
    for a in alphas:
        for b in betas:
            if 0 <= b < 90:
                az_idx = int(a // 45)
                el_idx = int(b // 30)
                grid_r6[el_idx, az_idx] += 1

im_r6 = axes[1].imshow(grid_r6, cmap="Blues", aspect="auto")
axes[1].set_xticks(range(8))
axes[1].set_xticklabels([f"{i*45}-{(i+1)*45}°" for i in range(8)], rotation=30)
axes[1].set_yticks(range(3))
axes[1].set_yticklabels(["0-30°", "30-60°", "60-90°"])
axes[1].set_title(f"R6: Viewpoint Coverage ({(grid_r6 > 0).sum()}/24 cells)")
for i in range(3):
    for j in range(8):
        axes[1].text(j, i, str(grid_r6[i, j]), ha="center", va="center", color="black")
plt.colorbar(im_r6, ax=axes[1])

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/R5_R6_scale_viewpoint.png", dpi=150)
plt.show()

In [ ]:
# ==================== CELL 5: R7 九宫格热力图 & R8/R9 统计 ====================
grid_r7 = np.zeros((3, 3), dtype=int)
for _, r in df_inst.iterrows():
    col = min(int(r["cx"] * 3), 2)
    row = min(int(r["cy"] * 3), 2)
    grid_r7[row, col] += 1

grid_r7_pct = grid_r7 / len(df_inst) * 100
fig, ax = plt.subplots(figsize=(5.5, 5))
im_r7 = ax.imshow(grid_r7_pct, cmap="YlOrRd")
ax.set_xticks([0, 1, 2]); ax.set_xticklabels(["Left", "Center", "Right"])
ax.set_yticks([0, 1, 2]); ax.set_yticklabels(["Top", "Middle", "Bottom"])
ax.set_title("R7: Box Centre 3x3 Grid Share (%)")
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{grid_r7_pct[i, j]:.1f}%\n({grid_r7[i, j]})", ha="center", va="center", color="black")
plt.colorbar(im_r7, ax=ax)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/R7_grid_heatmap.png", dpi=150)
plt.show()

In [ ]:
# ==================== CELL 6: R11 感知哈希近重复率检测 ====================
img_files = [os.path.join(DATASET_DIR, "images", name) for name in manifest["image"]]
hashes = [imagehash.phash(Image.open(p)) for p in img_files]
PHASH_THRESH = 5
dup_idx = set()
for i in range(len(hashes)):
    for j in range(i + 1, len(hashes)):
        if hashes[i] - hashes[j] <= PHASH_THRESH:
            dup_idx.add(i); dup_idx.add(j)
dup_rate = len(dup_idx) / len(hashes) * 100

In [ ]:
# ==================== CELL 7: R1-R11 总览汇总表 ====================
outer_cells = [grid_r7_pct[i, j] for i in range(3) for j in range(3) if not (i == 1 and j == 1)]
trunc_rate = df_inst["truncated"].mean() * 100
min_side = df_inst["short_side_px"].min()
max_poly_err = max(poly_box_diffs_px)

summary = pd.DataFrame([
    ["R1", "Dataset size & class", "1,000 imgs, 1 class", f"{len(manifest)} imgs, {data_cfg['names']}", len(manifest)==1000 and data_cfg['nc']==1],
    ["R2", "Instances per image", "70/10/8/6/6% (±3%)", "/".join(f"{v:.1f}%" for v in inst_shares), all(abs(inst_shares.iloc[k]-targets_r2[k])<=3 for k in range(5))],
    ["R3", "Image format", "Single resolution", f"{list(resolutions)}", len(resolutions)==1],
    ["R4", "Generation-mode mix", "M2: 300-330; Auto >=10%", dict(mode_counts), 300<=mode_counts['M2']<=330 and all(mode_counts[m]>=100 for m in ['M1','M3','M4','M5'])],
    ["R5", "Scale ladder", "30/40/30% (±5%)", f"{small_pct:.1f}/{med_pct:.1f}/{large_pct:.1f}%", all(abs(r5_meas[k]-r5_targ[k])<=5 for k in range(3))],
    ["R6", "Viewpoint coverage", "24/24 cells", f"{(grid_r6>0).sum()}/24 cells", (grid_r6>0).sum()==24],
    ["R7", "Position in frame", "Outer>=3%, Centre<=50%", f"MinOuter={min(outer_cells):.1f}%, Centre={grid_r7_pct[1,1]:.1f}%", min(outer_cells)>=3.0 and grid_r7_pct[1,1]<=50.0],
    ["R8", "Truncation share", "5–15% of instances", f"{trunc_rate:.2f}%", 5.0 <= trunc_rate <= 15.0],
    ["R9", "Minimum box size", ">= 16 px short side", f"{min_side:.1f} px", min_side >= 16.0],
    ["R10", "Label quality", "0 errors, diff <= 2 px", f"{len(r10_errors)} errors, max diff={max_poly_err:.2f} px", len(r10_errors)==0 and max_poly_err<=2.0],
    ["R11", "Near-duplicate rate", "<= 2% (pHash<=5)", f"{dup_rate:.2f}%", dup_rate <= 2.0],
], columns=["Req", "Metric", "Target", "Measured", "Pass"])

display(summary)